# 01 · Spatio-Temporal DeepSet — Handover Prediction (Production)
**Notebook:** `notebooks/modeling/01_temporal_deepset_production.ipynb`  
**Project root:** `../../`

### Architecture Overview
```
INPUT:  cells (B, K=10, T=25, F=3)   mask (B, K)   teacher_signals (B, S=5, 3)

══════════ ENCODER ════════════════════════════════════════════════════════════
  TimeDistributed(LSTM 64)           per-cell temporal encoder  → (B, K, 64)
  TimeDistributed(Dense 64×2)        Φ: per-cell embedding      → (B, K, 64)
  MaskedGlobalAveragePooling         z = Σ Φ(hᵢ)·mᵢ / Σ mᵢ   → (B, 64)
  Concat [Φ(hᵢ) ‖ z] + Dense 64     ρ: cell score in context   → (B, K, 64)
  Masked Softmax                     current step selection     → (B, K)

══════════ TEMPORAL DECODER (new) ════════════════════════════════════════════
  Context  [z ‖ max_pool(Φ)]         encoder summary            → (B, 128)
  Dense → h₀, c₀                    LSTM seed from context     → (B, 128)
  ┌── step s = 1 … 5 ─────────────────────────────────────────────────────┐
  │  LSTMCell([prev_probs ‖ prev_sig ‖ ctx])   hidden state   → (B, 128) │
  │  Attend(Φ, h_s)  + masked softmax          cell probs     → (B, K)   │
  │  WeightedSum(probs, Φ)  → reg MLP          signals        → (B, 3)   │
  │  teacher-force prev_sig during training,   autoregress at inference  │
  └───────────────────────────────────────────────────────────────────────┘
OUTPUT:
  cell_now     (B, K)      current-step cell selection probabilities
  cell_future  (B, S, K)  future-step cell selection probabilities
  sig_future   (B, S, 3)  future rsrp / sinr / load of selected cell
```

### Why Autoregressive?
Predicting 5 future steps with a flat dense head treats each step as independent.
Cell selection at `t+2` causally depends on which cell was selected at `t+1`
(hysteresis, handover delay, interference). The LSTM decoder explicitly models
this dependence: state `h_s` carries the forecasted trajectory into each next step.

### Loss Strategy
| Output | Loss | Notes |
|---|---|---|
| `cell_now` | Focal (γ=2, α=0.25) | Current-step cell classification |
| `cell_future` | Temporal Focal | Averaged over S=5 steps |
| `sig_future` | Huber (δ=1) | Robust to RSRP/SINR outliers |

In [31]:
# ─── Section 1 · Environment, Canonical Paths, GPU, MLflow ───────────────────
import os, sys, warnings, json, pickle, logging, datetime, gc, re
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing  import List, Tuple, Dict, Optional

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision
from sklearn.preprocessing      import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics            import (classification_report,
                                        confusion_matrix,
                                        top_k_accuracy_score)

sns.set_theme(style="whitegrid", font_scale=1.05)

# ── Project root ──────────────────────────────────────────────────────────────
_ROOT = Path("../../").resolve()

PATHS = dict(
    data    = _ROOT / "dataset" / "temporal_deepset_production_cache",
    models  = _ROOT / "models",
    tb_logs = _ROOT / "tb_logs" / "deepset_prod",
    metrics = _ROOT / "metrics" / "Temporal_deepset_production",
    mlruns  = _ROOT / "mlflow"  / "mlruns",
)
for p in PATHS.values():
    os.makedirs(str(p), exist_ok=True)

# ── Logging ───────────────────────────────────────────────────────────────────
_log_file = PATHS["metrics"] / "training.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s │ %(levelname)-8s │ %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout),
              logging.FileHandler(str(_log_file), mode="w")],
)
log = logging.getLogger("st_deepset")
log.info("Project root: %s", _ROOT)

# ── GPU ───────────────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
log.info("GPUs: %d", len(gpus))

# mixed_float16 disabled — the encoder→decoder dtype boundary was causing
# cascading NaN via float16 overflow; fp32 training is stable for this model size.
policy = mixed_precision.Policy("float32")
mixed_precision.set_global_policy(policy)
log.info("Precision: compute=%s  vars=%s", policy.compute_dtype, policy.variable_dtype)

# ── MLflow ────────────────────────────────────────────────────────────────────
_MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
try:
    import mlflow, mlflow.tensorflow, requests
    MLFLOW_OK = requests.get(f"{_MLFLOW_URI}/health", timeout=3).status_code == 200
    if MLFLOW_OK:
        mlflow.set_tracking_uri(_MLFLOW_URI)
        mlflow.set_experiment("temporal_deepset_production")
        log.info("MLflow → %s", _MLFLOW_URI)
    else:
        log.warning("MLflow unreachable — is SSH tunnel open?")
except Exception:
    MLFLOW_OK = False
    log.warning("mlflow not installed or unreachable.")

SEED = 42
tf.random.set_seed(SEED); np.random.seed(SEED)
log.info("TF %s | NumPy %s", tf.__version__, np.__version__)

09:20:09 │ INFO     │ Project root: /home/wassimmchichi/Downloads/Handover_projects
09:20:09 │ INFO     │ GPUs: 1
09:20:09 │ INFO     │ Precision: compute=float32  vars=float32
09:20:12 │ WARNING  │ mlflow not installed or unreachable.
09:20:12 │ INFO     │ TF 2.15.1 | NumPy 1.26.4


## Section 2 · Hyperparameters

In [32]:
# ─── Section 2 · Hyperparameters ─────────────────────────────────────────────
HP = dict(
    # ── Data ──────────────────────────────────────────────────────────────────
    MAX_CELLS    = 10,     # K: max candidate cells (including padding)
    OBS_STEPS    = 200,     # T: history window length (timesteps)
    N_FEATS      = 5,      # F: features per cell (rsrp, sinr, load, Δrsrp, Δsinr)
    PRED_STEPS   = 5,      # S: future steps to forecast

    # ── Loss ──────────────────────────────────────────────────────────────────
    LOSS_TYPE    = "focal",
    FOCAL_GAMMA  = 2.0,
    FOCAL_ALPHA  = 0.25,
    LABEL_SMOOTH = 0.1,
    REG_WEIGHT   = 0.15,   # weight for Huber regression (auxiliary task)
    FUTURE_WEIGHT= 2,    # weight for future cell-selection (reduced for stability)
    HUBER_DELTA  = 1.0,    # Huber loss delta for signal regression

    # ── Encoder ───────────────────────────────────────────────────────────────
    LSTM_UNITS   = 128,
    PHI_DIM      = 64,
    PHI_LAYERS   = 2,
    DROPOUT      = 0.25,

    # ── Decoder ───────────────────────────────────────────────────────────────
    DECODER_UNITS = 256,   # LSTM hidden size for autoregressive decoder (↑ capacity)
    ATTEND_DIM    = 128,   # attention hidden size for cell scoring in decoder (↑ capacity)

    # ── Training ──────────────────────────────────────────────────────────────
    BATCH_SIZE   = 64,
    EPOCHS       = 60,
    LR_INIT      = 1e-3,
    LR_WARMUP_EP = 4,
    LR_DECAY_EP  = 20,
)

ALL_LABELS = list(range(HP["MAX_CELLS"]))
log.info("HP loaded — PRED_STEPS=%d  DECODER_UNITS=%d",
         HP["PRED_STEPS"], HP["DECODER_UNITS"])

09:20:12 │ INFO     │ HP loaded — PRED_STEPS=5  DECODER_UNITS=256


## Section 3 · Loss Functions

Three loss components are combined:
- **`cell_now`**: Focal loss on current-step cell selection `(B, K)`
- **`cell_future`**: Temporal focal loss on future cell selections `(B, S, K)` — the same focal formula applied across the S time steps and then averaged. This enforces correct cell ordering across the forecast horizon.
- **`sig_future`**: Huber loss on (rsrp, sinr, load) forecasts `(B, S, 3)`. Huber is preferred over MSE because RSRP/SINR measurements contain occasional hardware-level spikes.

In [33]:
# ─── Section 3 · Loss Functions ───────────────────────────────────────────────

# ── 3a. Focal Loss — current step (B, K) ─────────────────────────────────────
def focal_loss(gamma: float = 2.0, alpha: float = 0.25):
    """
    Multi-class Focal Loss for one-hot targets.
      FL = -α · (1 - p_t)^γ · log(p_t)
    y_true: (B, C) one-hot
    y_pred: (B, C) softmax probabilities
    """
    def _loss(y_true, y_pred):
        y_pred = tf.clip_by_value(tf.cast(y_pred, tf.float32), 1e-7, 1.0 - 1e-7)
        y_true = tf.cast(y_true, tf.float32)
        ce     = -y_true * tf.math.log(y_pred)                         # (B, C)
        p_t    = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True) # (B, 1)
        fw     = alpha * tf.pow(1.0 - p_t, gamma)                       # (B, 1)
        return tf.reduce_mean(tf.reduce_sum(fw * ce, axis=-1))
    _loss.__name__ = f"focal_g{gamma}_a{alpha}"
    return _loss


# ── 3b. Temporal Focal Loss — future steps (B, S, K) ─────────────────────────
def temporal_focal_loss(gamma: float = 2.0, alpha: float = 0.25):
    """
    Focal loss extended to multi-step outputs.
    y_true: (B, S, K) one-hot per future step
    y_pred: (B, S, K) softmax probs per future step
    Loss averaged over B and S.
    """
    def _loss(y_true, y_pred):
        y_pred = tf.clip_by_value(tf.cast(y_pred, tf.float32), 1e-7, 1.0 - 1e-7)
        y_true = tf.cast(y_true, tf.float32)
        ce     = -y_true * tf.math.log(y_pred)                               # (B, S, K)
        p_t    = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True)       # (B, S, 1)
        fw     = alpha * tf.pow(1.0 - p_t, gamma)                             # (B, S, 1)
        step_loss = tf.reduce_sum(fw * ce, axis=-1)                           # (B, S)
        return tf.reduce_mean(step_loss)                                      # scalar
    _loss.__name__ = f"temporal_focal_g{gamma}_a{alpha}"
    return _loss


# ── 3c. Huber Loss — signal regression (B, S, 3) ─────────────────────────────
def huber_loss(delta: float = 1.0):
    """
    Huber loss for multi-output regression (rsrp, sinr, load).
    y_true: (B, S, 3)  y_pred: (B, S, 3)
    Robust to RSRP/SINR measurement spikes (outliers).
    """
    def _loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        err    = tf.abs(y_true - y_pred)
        h      = tf.where(err <= delta,
                          0.5 * err ** 2,
                          delta * (err - 0.5 * delta))
        return tf.reduce_mean(h)
    _loss.__name__ = f"huber_d{delta}"
    return _loss


# ── Build active loss functions ───────────────────────────────────────────────
FOCAL_FN       = focal_loss(HP["FOCAL_GAMMA"], HP["FOCAL_ALPHA"])
TEMP_FOCAL_FN  = temporal_focal_loss(HP["FOCAL_GAMMA"], HP["FOCAL_ALPHA"])
HUBER_FN       = huber_loss(HP["HUBER_DELTA"])

log.info("Losses: %s | %s | %s",
         FOCAL_FN.__name__, TEMP_FOCAL_FN.__name__, HUBER_FN.__name__)

09:20:12 │ INFO     │ Losses: focal_g2.0_a0.25 | temporal_focal_g2.0_a0.25 | huber_d1.0


## Section 4 · Data Pipeline

### Preprocessing for list-like columns
The following string/list columns need parsing:
- `nb_cell_ids`, `nb_cell_types`, `nb_net_types` → integer lists (cell identifiers)
- `nb_rsrps`, `nb_sinrs`, `nb_loads`, `nb_tp_ests`, `nb_dists_m`, `nb_path_losses_db`, `nb_scores` → float lists

Each is stored as a Python string like `"[val1, val2, ...]"`. We parse, pad/truncate to `MAX_CELLS=10`, and convert to numpy arrays.

### Multi-step target generation
For each window `[t-T, t)` we generate **5 future labels**:
- `y_cell_now`: current optimal cell index in the shuffled order (same as v1)
- `y_cell_future[s]`: optimal cell index at `t+s` mapped into the **same** shuffled order `p` (neighbor set assumed stable over 1 s)
- `y_signals[s]`: (`optimal_cell_rsrp`, `optimal_cell_sinr`, `optimal_cell_load`) at `t+s` — scalar columns, no permutation needed

Signal targets are normalized with a separate `StandardScaler` fitted on the training split only.

In [34]:
import re
import gc
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# ─── Memory-Efficient Parsing Helper ──────────────────────────────────────────

def parse_to_numpy(series, max_len, fill_val=0.0, is_int=False):
    """Parses string series directly into a pre-allocated 2D NumPy array to bypass RAM spikes."""
    dtype = np.int32 if is_int else np.float32
    out_arr = np.full((len(series), max_len), fill_val, dtype=dtype)
    
    # Fast string processing loop directly to primitive values
    for idx, s in enumerate(series.fillna("").astype(str).values):
        if not s or s == 'nan':
            continue
        cleaned = s.replace('[', '').replace(']', '').replace(',', ' ').replace(';', ' ')
        parts = cleaned.split()
        for i, p in enumerate(parts[:max_len]):
            if p in ('', 'nan', 'none', 'None'):
                continue
            try:
                out_arr[idx, i] = int(float(p)) if is_int else float(p)
            except ValueError:
                pass
    return out_arr


# ─── Main Pipeline ────────────────────────────────────────────────────────────

def load_and_create_datasets(root_dir):
    """
    Memory-optimized pipeline with exact same structural variables and outputs.
    Pre-allocates flat tensors and mutates values in-place to avoid OOM crashes.
    """
    raw_path = root_dir / "dataset" / "raw" / "handover_dataset.csv"
    log.info("Loading raw data from %s ...", raw_path)
    df = pd.read_csv(raw_path, low_memory=False)
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="mixed")
    df.sort_values(["ue_id", "timestamp"], inplace=True)
    df.reset_index(drop=True, inplace=True)

    K = HP["MAX_CELLS"]
    T_win = HP["OBS_STEPS"]
    S = HP["PRED_STEPS"]
    F_feats = HP.get("N_FEATS", 5) # Adapts to your feature footprint dynamically
    rng = np.random.default_rng(SEED)

    # 1. Parse text arrays straight to lightweight NumPy representations
    log.info("Parsing columns to low-overhead NumPy blocks...")
    nb_ids_arr   = parse_to_numpy(df["nb_cell_ids"], K, fill_val=0, is_int=True)
    nb_rsrps_arr = parse_to_numpy(df["nb_rsrps"], K, fill_val=0.0, is_int=False)
    nb_sinrs_arr = parse_to_numpy(df["nb_sinrs"], K, fill_val=0.0, is_int=False)
    nb_loads_arr = parse_to_numpy(df["nb_loads"], K, fill_val=0.0, is_int=False)

    opt_ids   = pd.to_numeric(df["optimal_cell_id"], errors="coerce").fillna(0).astype(np.int32).values
    sig_rsrp  = pd.to_numeric(df["optimal_cell_rsrp"], errors="coerce").fillna(0.0).astype(np.float32).values
    sig_sinr  = pd.to_numeric(df["optimal_cell_sinr"], errors="coerce").fillna(0.0).astype(np.float32).values
    sig_load  = pd.to_numeric(df["optimal_cell_load"], errors="coerce").fillna(0.0).astype(np.float32).values

    # Clean up original string storage columns immediately to reclaim space
    df.drop(columns=["nb_cell_ids", "nb_rsrps", "nb_sinrs", "nb_loads"], errors='ignore', inplace=True)
    gc.collect()

    # 2. Compute window sizes for upfront memory pre-allocation
    log.info("Calculating exact structure layouts to pre-allocate memory...")
    ue_groups = df.groupby("ue_id", sort=False).indices
    total_windows = 0
    valid_ues = {}
    
    for ue_id, idxs in ue_groups.items():
        n_row = len(idxs)
        n_wind = n_row - T_win - S
        if n_wind > 0:
            total_windows += n_wind
            valid_ues[ue_id] = (idxs, n_wind)

    # 3. Direct Allocation of Final Tensors (No hidden double-allocations via .append)
    X = np.zeros((total_windows, K, T_win, F_feats), dtype=np.float32)
    M = np.zeros((total_windows, K), dtype=np.float32)
    y = np.zeros((total_windows,), dtype=np.int32)
    yfc = np.zeros((total_windows, S), dtype=np.int32)
    ys_raw = np.zeros((total_windows, S, 3), dtype=np.float32)
    groups = np.empty((total_windows,), dtype=object)

    log.info("Processing sliding windows inline across pre-allocated block...")
    idx_ptr = 0
    
    for ue_id, (idxs, n_wind) in valid_ues.items():
        n_row = len(idxs)
        
        # Pull vectorized views smoothly
        ue_ids = nb_ids_arr[idxs]
        ue_rsrps = nb_rsrps_arr[idxs]
        ue_sinrs = nb_sinrs_arr[idxs]
        ue_loads = nb_loads_arr[idxs]
        ue_opt = opt_ids[idxs]
        ue_s_rsrp = sig_rsrp[idxs]
        ue_s_sinr = sig_sinr[idxs]
        ue_s_load = sig_load[idxs]

        ue_perm = rng.permutation(K)
        rs = ue_rsrps[:, ue_perm]
        sn = ue_sinrs[:, ue_perm]
        ld = ue_loads[:, ue_perm]
        nb_ids_col = ue_ids[:, ue_perm]

        cell_feat = np.zeros((n_row, K, 5), dtype=np.float32)
        cell_feat[:, :, 0] = rs
        cell_feat[:, :, 1] = sn
        cell_feat[:, :, 2] = ld
        if n_row > 1:
            cell_feat[1:, :, 3] = rs[1:] - rs[:-1]
            cell_feat[1:, :, 4] = sn[1:] - sn[:-1]

        for t in range(T_win, n_row - S):
            X_w = cell_feat[t - T_win : t]
            p = rng.permutation(K)
            X_ws = X_w[:, p, :].transpose(1, 0, 2)

            M_w = (X_ws[:, -1, 0] != 0.0).astype(np.float32)

            try:
                orig_idx = list(nb_ids_col[t]).index(int(ue_opt[t]))
                y_now = int(np.where(p == orig_idx)[0][0])
            except (ValueError, IndexError):
                y_now = -1

            fut_cells = np.zeros(S, dtype=np.int32)
            fut_signals = np.zeros((S, 3), dtype=np.float32)
            
            for s_idx in range(S):
                ft = t + s_idx + 1
                try:
                    orig_idx_f = list(nb_ids_col[t]).index(int(ue_opt[ft]))
                    lbl = int(np.where(p == orig_idx_f)[0][0])
                except (ValueError, IndexError):
                    lbl = -1
                fut_cells[s_idx] = y_now if lbl == -1 else lbl
                fut_signals[s_idx, 0] = ue_s_rsrp[ft]
                fut_signals[s_idx, 1] = ue_s_sinr[ft]
                fut_signals[s_idx, 2] = ue_s_load[ft]

            X[idx_ptr] = X_ws[:, :, :F_feats]
            M[idx_ptr] = M_w
            y[idx_ptr] = y_now
            yfc[idx_ptr] = fut_cells
            ys_raw[idx_ptr] = fut_signals
            groups[idx_ptr] = ue_id
            idx_ptr += 1

    # Clear intermediate arrays completely before splitting and scaling
    del nb_ids_arr, nb_rsrps_arr, nb_sinrs_arr, nb_loads_arr, df
    gc.collect()

    # 4. Generate data split configurations
    ue_list = np.unique(groups)
    rng.shuffle(ue_list)
    n_te = int(len(ue_list) * 0.15)
    n_va = int(len(ue_list) * 0.15)
    ue_te = set(ue_list[:n_te])
    ue_va = set(ue_list[n_te : n_te + n_va])

    idx_tr = np.where([u not in ue_te and u not in ue_va for u in groups])[0]
    idx_va = np.where([u in ue_va for u in groups])[0]
    idx_te = np.where([u in ue_te for u in groups])[0]

    # 5. Scaler processing directly in-place to ensure memory efficiency
    log.info("Fitting and transforming scalers in-place...")
    cell_scaler = StandardScaler()
    
    tr_mask = np.zeros(len(X), dtype=bool)
    tr_mask[idx_tr] = True
    
    # Target only active training elements without duplicating arrays
    valid_train_idx = np.where((M == 1.0) & tr_mask[:, np.newaxis])
    cell_scaler.fit(X[valid_train_idx[0], valid_train_idx[1]].reshape(-1, F_feats))

    # Apply scaling transformation directly onto target locations inside X
    all_valid_idx = np.where(M == 1.0)
    flat_feats = X[all_valid_idx[0], all_valid_idx[1]].reshape(-1, F_feats)
    X[all_valid_idx[0], all_valid_idx[1]] = cell_scaler.transform(flat_feats).reshape(-1, T_win, F_feats)
    
    del flat_feats
    gc.collect()

    # Normalize regression coordinates smoothly
    sig_scaler = StandardScaler()
    sig_scaler.fit(ys_raw[idx_tr].reshape(-1, 3))
    
    ys_shape = ys_raw.shape
    ys_raw = sig_scaler.transform(ys_raw.reshape(-1, 3)).reshape(ys_shape)

    log.info("Processing complete. Delivering arrays clean.")

    # Splitting creates isolated variable objects; parent variables clear out of scope
    return (
        X[idx_tr], M[idx_tr], y[idx_tr], yfc[idx_tr], ys_raw[idx_tr],
        X[idx_va], M[idx_va], y[idx_va], yfc[idx_va], ys_raw[idx_va],
        X[idx_te], M[idx_te], y[idx_te], yfc[idx_te], ys_raw[idx_te],
        cell_scaler, sig_scaler,
    )


# ── Run pipeline ──────────────────────────────────────────────────────────────
(
    X_tr, M_tr, y_tr, yfc_tr, ys_tr,
    X_va, M_va, y_va, yfc_va, ys_va,
    X_te, M_te, y_te, yfc_te, ys_te,
    cell_scaler, sig_scaler,
) = load_and_create_datasets(_ROOT)

log.info("X_tr: %s  M_tr: %s  y_tr: %s  yfc_tr: %s  ys_tr: %s",
         X_tr.shape, M_tr.shape, y_tr.shape, yfc_tr.shape, ys_tr.shape)

09:20:12 │ INFO     │ Loading raw data from /home/wassimmchichi/Downloads/Handover_projects/dataset/raw/handover_dataset.csv ...
09:20:14 │ INFO     │ Parsing columns to low-overhead NumPy blocks...
09:20:16 │ INFO     │ Calculating exact structure layouts to pre-allocate memory...
09:20:16 │ INFO     │ Processing sliding windows inline across pre-allocated block...
09:20:21 │ INFO     │ Fitting and transforming scalers in-place...
09:20:28 │ INFO     │ Processing complete. Delivering arrays clean.
09:20:28 │ INFO     │ X_tr: (20160, 10, 200, 5)  M_tr: (20160, 10)  y_tr: (20160,)  yfc_tr: (20160, 5)  ys_tr: (20160, 5, 3)


In [35]:
# ─── Section 4b · tf.data Dataset Creation ────────────────────────────────────
#
# Each sample is a dict of inputs:
#   cells           (K, T, F)   cell feature windows
#   mask            (K,)        valid-cell mask
#   teacher_signals (S, 3)      future signal targets (used only in training)
#
# Each label is a dict of outputs:
#   cell_now    (K,)   one-hot — current-step optimal cell
#   cell_future (S, K) one-hot — future optimal cells
#   sig_future  (S, 3) float   — future normalised rsrp/sinr/load

K = HP["MAX_CELLS"]
S = HP["PRED_STEPS"]

cw_vals         = compute_class_weight("balanced", classes=np.unique(y_tr), y=y_tr)
CLASS_WEIGHT    = {int(c): float(w) for c, w in enumerate(cw_vals)}

def make_ds(X, M, y, yfc, ys, cw=None, shuffle=False):
    # One-hot encode cell labels
    y_oh   = tf.one_hot(y,   depth=K).numpy().astype(np.float32)  # (N, K)
    yfc_oh = tf.one_hot(yfc, depth=K).numpy().astype(np.float32)  # (N, S, K)

    inputs = {
        "cells"           : X,
        "mask"            : M,
        "teacher_signals" : ys.astype(np.float32),  # (N, S, 3)
    }
    labels = {
        "cell_now"    : y_oh,
        "cell_future" : yfc_oh,
        "sig_future"  : ys.astype(np.float32),
    }
    
    # FIX: Force dataset initialization to happen on host memory (CPU)
    with tf.device('/CPU:0'):
        if cw is not None:
            sw = np.array([cw[int(lbl)] for lbl in y], dtype=np.float32)
            sample_weights = {"cell_now": sw}
            ds = tf.data.Dataset.from_tensor_slices((inputs, labels, sample_weights))
        else:
            ds = tf.data.Dataset.from_tensor_slices((inputs, labels))
            
        if shuffle:
            ds = ds.shuffle(len(y), seed=SEED, reshuffle_each_iteration=True)
            
    return ds.batch(HP["BATCH_SIZE"]).prefetch(tf.data.AUTOTUNE)


ds_tr = make_ds(X_tr, M_tr, y_tr, yfc_tr, ys_tr, cw=CLASS_WEIGHT, shuffle=True)
ds_va = make_ds(X_va, M_va, y_va, yfc_va, ys_va)
ds_te = make_ds(X_te, M_te, y_te, yfc_te, ys_te)

steps_per_epoch = int(np.ceil(len(X_tr) / HP["BATCH_SIZE"]))

log.info("Datasets ready. train batches:%d  class_weight keys:%d",
         len(ds_tr), len(CLASS_WEIGHT))

# Peek at one batch to confirm shapes
for batch in ds_tr.take(1):
    inp_b, lbl_b = batch[0], batch[1]
    for k, v in inp_b.items():
        log.info("  input  %-20s → %s", k, v.shape)
    for k, v in lbl_b.items():
        log.info("  label  %-20s → %s", k, v.shape)

09:20:29 │ INFO     │ Datasets ready. train batches:315  class_weight keys:10
09:20:30 │ INFO     │   input  cells                → (64, 10, 200, 5)
09:20:30 │ INFO     │   input  mask                 → (64, 10)
09:20:30 │ INFO     │   input  teacher_signals      → (64, 5, 3)
09:20:30 │ INFO     │   label  cell_now             → (64, 10)
09:20:30 │ INFO     │   label  cell_future          → (64, 5, 10)
09:20:30 │ INFO     │   label  sig_future           → (64, 5, 3)


## Section 5 · Custom Keras Layers

### `MaskedGlobalAveragePooling`
Standard `GlobalAveragePooling1D` divides by `MAX_CELLS=10` even when only 7 real cells are present, diluting the context vector by 30%. This layer uses the actual real-cell count as denominator.

### `AutoregressiveDecoder`
The key new component. At each of the S=5 forecast steps:
1. **Input** = `[prev_cell_probs ‖ prev_signals ‖ encoder_ctx]` (projected to `DECODER_UNITS`)
2. **LSTMCell** advances the hidden state `h_s`
3. **Cell attention**: `h_s` is tiled over the K cells, concatenated with `Φ(h_i)`, and scored → masked softmax → cell probabilities
4. **Signal regression**: weighted average of `Φ(h_i)` by cell probs → dense regression → (rsrp, sinr, load)
5. **Teacher forcing** (training only): next step receives ground-truth signals, not predicted ones

In [36]:
# ─── Section 5 · Custom Keras Layers ─────────────────────────────────────────


class MaskedGlobalAveragePooling(keras.layers.Layer):
    """
    DeepSet aggregator: z = Σ Φ(hᵢ)·maskᵢ / Σ maskᵢ

    Args:
        phi  : (B, C, D)  per-cell embeddings
        mask : (B, C, 1)  real-cell indicator (1=real, 0=padded)
    Returns:
        z    : (B, D)     masked average pooling
    """
    def call(self, phi, mask):
        summed = tf.reduce_sum(phi * mask, axis=1)                # (B, D)
        count  = tf.maximum(tf.reduce_sum(mask, axis=1), 1e-8)    # (B, 1)
        return summed / count

    def get_config(self):
        return super().get_config()


class AutoregressiveDecoder(keras.layers.Layer):
    """
    LSTM-based autoregressive temporal decoder for multi-step cell forecasting.

    At each step s ∈ {1 … pred_steps}:
      1. Project [prev_probs ‖ prev_signals ‖ encoder_ctx] → decoder input
      2. LSTMCell(input, [h, c]) → h_s
      3. Attention score: score_k = Dense(Φ_k ‖ h_s) → masked softmax → probs_s
      4. Regression: weighted_phi = Σ probs_k · Φ_k → Dense → signals_s
      5. Teacher forcing (training=True, teacher_signals provided):
           prev_signals ← teacher_signals[:, s-1, :]
         Autoregressive (training=False):
           prev_signals ← signals_s

    Args:
        pred_steps    : number of future steps S
        max_cells     : K (must equal number of cells in phi)
        decoder_units : LSTM hidden size
        attend_dim    : hidden size of the attention MLP

    Call args:
        ctx             : (B, ctx_dim)    encoder summary vector
        phi             : (B, K, phi_dim) per-cell embeddings from encoder
        mask            : (B, K)          valid-cell mask
        teacher_signals : (B, S, 3) or None
        training        : bool

    Returns:
        cell_preds   : (B, S, K)   future cell probability distributions
        signal_preds : (B, S, 3)   future (rsrp, sinr, load) predictions
    """

    def __init__(self, pred_steps, max_cells, decoder_units, attend_dim, **kwargs):
        super().__init__(**kwargs)
        self.pred_steps    = pred_steps
        self.max_cells     = max_cells
        self.decoder_units = decoder_units
        self.attend_dim    = attend_dim

        # All sub-layers are forced to float32 regardless of the global
        # mixed_float16 policy, which would otherwise cast every input to fp16.
        fp32 = "float32"

        self.h_init = keras.layers.Dense(decoder_units, activation="tanh",
                                          dtype=fp32, name="dec_h_init")
        self.c_init = keras.layers.Dense(decoder_units, activation="tanh",
                                          dtype=fp32, name="dec_c_init")
        self.lstm_cell  = keras.layers.LSTMCell(decoder_units,
                                                 dtype=fp32, name="dec_lstm_cell")
        self.input_proj = keras.layers.Dense(decoder_units, activation="relu",
                                              dtype=fp32, name="dec_input_proj")
        self.attend_dense = keras.layers.Dense(attend_dim, activation="relu",
                                                dtype=fp32, name="dec_attend")
        self.attend_score = keras.layers.Dense(1,
                                                dtype=fp32, name="dec_attend_score")
        self.reg_dense1 = keras.layers.Dense(64, activation="relu",
                                              dtype=fp32, name="dec_reg1")
        self.reg_out    = keras.layers.Dense(3,
                                              dtype=fp32, name="dec_reg_out")


    def call(self, ctx, phi, mask, teacher_signals=None, training=False):
        B = tf.shape(ctx)[0]
        K = self.max_cells

        # Cast inputs to float32 — sub-layers are already fp32 (set in __init__).
        ctx  = tf.cast(ctx,  tf.float32)
        phi  = tf.cast(phi,  tf.float32)
        mask = tf.cast(mask, tf.float32)
        if teacher_signals is not None:
            teacher_signals = tf.cast(teacher_signals, tf.float32)

        # ── Initialise decoder state from encoder context ──────────────────────
        h = self.h_init(ctx)   # (B, decoder_units) float32
        c = self.c_init(ctx)   # (B, decoder_units) float32

        # ── Seed inputs for step 0 ────────────────────────────────────────────
        prev_probs   = tf.zeros((B, K), dtype=tf.float32)
        prev_signals = tf.zeros((B, 3), dtype=tf.float32)

        all_cell_preds   = []
        all_signal_preds = []

        for step in range(self.pred_steps):
            # ── Decoder input ─────────────────────────────────────────────────
            dec_in      = tf.concat([prev_probs, prev_signals, ctx], axis=-1)
            dec_in_proj = self.input_proj(dec_in)       # (B, decoder_units) fp32

            # ── LSTMCell step ─────────────────────────────────────────────────
            output, [h, c] = self.lstm_cell(dec_in_proj, [h, c])

            # ── Attention scores (scaled, float32) ────────────────────────────
            h_tiled   = tf.tile(tf.expand_dims(output, 1), [1, K, 1])
            attend_in = tf.concat([phi, h_tiled], axis=-1)
            attend_h  = self.attend_dense(attend_in)
            scores    = tf.squeeze(self.attend_score(attend_h), axis=-1)  # (B,K)

            scores_scaled = scores / tf.math.sqrt(float(self.attend_dim))
            neg_inf    = (1.0 - mask) * (-1e4)
            cell_probs = tf.nn.softmax(scores_scaled + neg_inf, axis=-1)
            all_cell_preds.append(cell_probs)

            # ── Signal regression ─────────────────────────────────────────────
            selected_phi = tf.reduce_sum(
                tf.expand_dims(cell_probs, -1) * phi, axis=1)
            reg_in   = tf.concat([output, selected_phi], axis=-1)
            reg_h    = self.reg_dense1(reg_in)
            signals  = self.reg_out(reg_h)
            signals  = tf.clip_by_value(signals, -20.0, 20.0)
            all_signal_preds.append(signals)

            # ── Teacher forcing vs autoregressive ─────────────────────────────
            prev_probs = cell_probs
            if training and teacher_signals is not None:
                prev_signals = teacher_signals[:, step, :]
            else:
                prev_signals = signals

        cell_preds   = tf.stack(all_cell_preds,   axis=1)  # (B, S, K)
        signal_preds = tf.stack(all_signal_preds, axis=1)  # (B, S, 3)
        return cell_preds, signal_preds


    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            "pred_steps"   : self.pred_steps,
            "max_cells"    : self.max_cells,
            "decoder_units": self.decoder_units,
            "attend_dim"   : self.attend_dim,
        })
        return cfg


print("Custom layers defined: MaskedGlobalAveragePooling, AutoregressiveDecoder")

Custom layers defined: MaskedGlobalAveragePooling, AutoregressiveDecoder


## Section 6 · Learning Rate Schedule

In [37]:
# ─── Section 6 · Learning-Rate Schedule ───────────────────────────────────────

class WarmUpCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    """Linear warm-up → cosine decay → flat floor. Step-based."""
    def __init__(self, lr_max, lr_min, warmup_steps, decay_steps):
        super().__init__()
        self.lr_max = float(lr_max); self.lr_min = float(lr_min)
        self.warmup_steps = float(warmup_steps)
        self.decay_steps  = float(decay_steps)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        warmup = self.lr_max * step / tf.maximum(self.warmup_steps, 1.0)
        cos    = self.lr_min + 0.5 * (self.lr_max - self.lr_min) * (
            1.0 + tf.cos(np.pi *
                tf.minimum(step - self.warmup_steps, self.decay_steps)
                / self.decay_steps))
        return tf.where(step < self.warmup_steps, warmup, cos)

    def get_config(self):
        return {"lr_max": self.lr_max, "lr_min": self.lr_min,
                "warmup_steps": self.warmup_steps, "decay_steps": self.decay_steps}


warmup_steps = HP["LR_WARMUP_EP"] * steps_per_epoch
decay_steps  = HP["LR_DECAY_EP"]  * steps_per_epoch
lr_sched     = WarmUpCosineDecay(HP["LR_INIT"], HP["LR_INIT"] * 0.01,
                                  warmup_steps, decay_steps)

# Plot
ep_lr = [float(lr_sched(e * steps_per_epoch)) for e in range(HP["EPOCHS"])]
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(ep_lr, lw=2, color="#2196F3")
ax.axvline(HP["LR_WARMUP_EP"], color="orange", ls="--", lw=1.2,
           label=f"Warm-up end (ep {HP['LR_WARMUP_EP']})")
ax.axvline(HP["LR_WARMUP_EP"] + HP["LR_DECAY_EP"], color="red",
           ls="--", lw=1.2, label="Cosine end")
ax.set(xlabel="Epoch", ylabel="LR", title="WarmUpCosineDecay")
ax.legend(fontsize=9); ax.grid(alpha=0.4)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1e}"))
plt.tight_layout()
_out = PATHS["metrics"] / "lr_schedule.png"
plt.savefig(str(_out), dpi=150, bbox_inches="tight"); plt.close()
log.info("LR schedule saved: %s", _out)

09:20:30 │ INFO     │ LR schedule saved: /home/wassimmchichi/Downloads/Handover_projects/metrics/Temporal_deepset_production/lr_schedule.png


## Section 7 · Spatio-Temporal DeepSet Model

### Tensor shapes at each stage
```
Input
  inp_cells          (B, K=10, T=25, F=3)
  inp_mask           (B, K=10)
  inp_teacher        (B, S=5,  3)

Encoder
  td_lstm            (B, K, 64)         shared LSTM per cell over T
  phi_0..1           (B, K, 64)         Φ: non-linear per-cell embedding
  mask_exp           (B, K, 1)          broadcasted mask
  z (pool)           (B, 64)            masked global context
  z_tiled            (B, K, 64)         context replicated per cell
  rho                (B, K, 64)         per-cell + global scoring
  logits             (B, K)             raw scores
  cell_now           (B, K)             masked softmax → current probs

Decoder context
  max_phi            (B, 64)            max-pool over valid cells
  ctx                (B, 128)           [z ‖ max_phi] encoder summary

Decoder (AutoregressiveDecoder)
  h_0, c_0           (B, 128)           LSTM seed from ctx
  per step s:
    dec_in           (B, K+3+128)       [prev_probs ‖ prev_sig ‖ ctx]
    dec_in_proj      (B, 128)           linear projection
    h_s              (B, 128)           LSTMCell output
    attend_in        (B, K, 64+128)     [Φ ‖ h_s tiled]
    scores           (B, K)             attention scores
    cell_probs_s     (B, K)             masked softmax
    selected_phi     (B, 64)            Σ probs_k · Φ_k
    signals_s        (B, 3)             rsrp, sinr, load

Output
  cell_now           (B, K)             current step cell probs
  cell_future        (B, S, K)          future cell probs
  sig_future         (B, S, 3)          future signals (normalised)
```

In [38]:
# ─── Section 7 · Spatio-Temporal DeepSet Model ────────────────────────────────

def build_spatiotemporal_deepset(hp: dict) -> keras.Model:
    """
    Build the full spatio-temporal model with:
      - Per-cell temporal encoder (shared LSTM + Phi MLP + masked pooling)
      - Current-step masked softmax cell head
      - Autoregressive temporal decoder for S future steps
    """
    K  = hp["MAX_CELLS"]
    T  = hp["OBS_STEPS"]
    F  = hp["N_FEATS"]
    D  = hp["PHI_DIM"]
    S  = hp["PRED_STEPS"]

    # ── Inputs ────────────────────────────────────────────────────────────────
    inp_cells   = keras.Input((K, T, F), name="cells",           dtype="float32")
    inp_mask    = keras.Input((K,),      name="mask",            dtype="float32")
    inp_teacher = keras.Input((S, 3),    name="teacher_signals", dtype="float32")
    # inp_teacher is only used during training (teacher forcing).
    # During inference pass zeros for this input.

    # ══════════════════════════════════════════════════════════════════════════
    # ENCODER
    # ══════════════════════════════════════════════════════════════════════════

    # Stage 1 — Shared LSTM encoder per cell
    # TimeDistributed applies the same LSTM to every cell independently.
    # The LSTM reads the T-step time series [rsrp_t, sinr_t, load_t] for cell k.
    # Output: final hidden state hᵢ for each cell.
    trend = layers.TimeDistributed(
        layers.LSTM(hp["LSTM_UNITS"], return_sequences=False),
        name="td_lstm"
    )(inp_cells)   # (B, K, lstm_units)

    # Stage 2 — Φ: shared non-linear per-cell embedding
    # PHI_LAYERS stacked Dense+ReLU+Dropout applied to every cell identically.
    phi = trend
    for i in range(hp["PHI_LAYERS"]):
        phi = layers.TimeDistributed(
            layers.Dense(D, activation="relu"),
            name=f"phi_{i}"
        )(phi)
        phi = layers.TimeDistributed(
            layers.Dropout(hp["DROPOUT"]),
            name=f"phi_drop_{i}"
        )(phi)
    # phi: (B, K, D=64)

    # Stage 3 — Masked global average pool → context vector z
    # z = Σᵢ Φ(hᵢ) · maskᵢ / Σᵢ maskᵢ
    mask_exp = layers.Reshape((K, 1), name="mask_exp")(inp_mask)  # (B, K, 1)
    z        = MaskedGlobalAveragePooling(name="masked_pool")(phi, mask_exp)  # (B, D)

    # Stage 4 — ρ: per-cell scoring in global context
    # Each cell sees its own embedding Φ(hᵢ) concatenated with the global z.
    z_tiled = layers.RepeatVector(K, name="z_tile")(z)             # (B, K, D)
    rho     = layers.Concatenate(axis=-1, name="rho_cat")([phi, z_tiled])  # (B, K, 2D)
    rho     = layers.TimeDistributed(
        layers.Dense(D, activation="relu"), name="rho")(rho)
    rho     = layers.TimeDistributed(
        layers.Dropout(hp["DROPOUT"]), name="rho_drop")(rho)
    # rho: (B, K, D)

    # Stage 5 — Masked softmax → current-step cell probabilities
    logits    = layers.TimeDistributed(layers.Dense(1), name="scorer")(rho)
    logits    = layers.Reshape((K,), name="logits")(logits)        # (B, K)
    # Add -inf to padded cells before softmax
    pad_mask  = layers.Lambda(
        lambda m: tf.cast((1.0 - m), tf.float32) * (-1e9),
        name="pad_bias"
    )(inp_mask)                                                    # (B, K)
    cell_now  = layers.Softmax(dtype="float32", name="cell_now")(
        layers.Add(name="masked_logits")([logits, pad_mask]))      # (B, K)

    # ══════════════════════════════════════════════════════════════════════════
    # ENCODER SUMMARY → DECODER CONTEXT
    # ══════════════════════════════════════════════════════════════════════════

    # Max-pool over the valid cells to get a complementary summary.
    # (z captures the average cell; max_phi captures the best cell.)
    # We mask the padded cells to -inf before max-pooling.
    phi_masked  = layers.Add(name="phi_pad_bias")(
        [phi, layers.Lambda(
            lambda m: tf.cast((1.0 - m), tf.float32) * (-1e9),
            name="phi_bias_mask"
        )(mask_exp)])                                              # (B, K, D)
    max_phi     = layers.GlobalMaxPooling1D(name="max_phi")(phi_masked)  # (B, D)

    # ctx: encoder summary = [z ‖ max_phi]
    ctx = layers.Concatenate(name="enc_ctx")([z, max_phi])         # (B, 2D=128)

    # ══════════════════════════════════════════════════════════════════════════
    # TEMPORAL DECODER
    # ══════════════════════════════════════════════════════════════════════════

    decoder = AutoregressiveDecoder(
        pred_steps    = hp["PRED_STEPS"],
        max_cells     = K,
        decoder_units = hp["DECODER_UNITS"],
        attend_dim    = hp["ATTEND_DIM"],
        name          = "autoregressive_decoder",
    )

    # The decoder uses phi (per-cell embeddings) and ctx (encoder summary).
    # During training it receives teacher_signals; during inference zeros are fine
    # because the training=False branch of AutoregressiveDecoder ignores them.
    cell_future, sig_future = decoder(
        ctx, phi, inp_mask, inp_teacher
    )   # (B, S, K), (B, S, 3)

    # ── Name the outputs explicitly for loss dict ──────────────────────────────
    cell_future = layers.Lambda(lambda x: x, name="cell_future")(cell_future)
    sig_future  = layers.Lambda(lambda x: x, name="sig_future")(sig_future)

    # ── Build model ───────────────────────────────────────────────────────────
    model = keras.Model(
        inputs  = [inp_cells, inp_mask, inp_teacher],
        outputs = {"cell_now": cell_now,
                   "cell_future": cell_future,
                   "sig_future":  sig_future},
        name    = "SpatioTemporalDeepSet",
    )
    return model


# ── Build & compile ───────────────────────s────────────────────────────────────
model = build_spatiotemporal_deepset(HP)

model.compile(
    optimizer    = keras.optimizers.Adam(learning_rate=lr_sched, clipnorm=1.0),
    loss         = {
        "cell_now"    : FOCAL_FN,
        "cell_future" : TEMP_FOCAL_FN,
        "sig_future"  : HUBER_FN,
    },
    loss_weights = {
        "cell_now"    : 1.0,
        "cell_future" : HP["FUTURE_WEIGHT"],
        "sig_future"  : HP["REG_WEIGHT"],
    },
    metrics      = {
        "cell_now"    : [
            keras.metrics.CategoricalAccuracy(name="top1"),
            keras.metrics.TopKCategoricalAccuracy(k=3, name="top3"),
        ],
        "cell_future" : [
            keras.metrics.CategoricalAccuracy(name="fut_top1"),
        ],
        "sig_future"  : [
            keras.metrics.MeanAbsoluteError(name="sig_mae"),
        ],
    },
)

model.summary(line_length=96, expand_nested=False)
log.info("Total parameters: %d", model.count_params())

Model: "SpatioTemporalDeepSet"
________________________________________________________________________________________________
 Layer (type)               Output Shape                 Param #   Connected to                 
 cells (InputLayer)         [(None, 10, 200, 5)]         0         []                           
                                                                                                
 td_lstm (TimeDistributed)  (None, 10, 128)              68608     ['cells[0][0]']              
                                                                                                
 phi_0 (TimeDistributed)    (None, 10, 64)               8256      ['td_lstm[0][0]']            
                                                                                                
 phi_drop_0 (TimeDistribut  (None, 10, 64)               0         ['phi_0[0][0]']              
 ed)                                                                                            

## Section 8 · Training

### Training strategy
- **Teacher forcing** is active during training: the decoder sees ground-truth signal values at each step, which greatly stabilises early training of the LSTM decoder.
- **Autoregressive evaluation**: during validation and test the model runs fully autoregressively (no teacher forcing), which is the production inference mode.
- **Primary monitor**: `val_cell_now_top3` (top-3 accuracy on current-step cell selection) — consistent with the v1 notebook so checkpoints are comparable.
- **Class weights**: applied to the `cell_now` head only (Keras does not support per-head class weights natively; future work can apply per-head sample weights).

In [39]:
# ─── Section 8 · Training ─────────────────────────────────────────────────────

MONITOR   = "val_cell_future_fut_top1"
CKPT_PATH = str(PATHS["models"] / "best_st_deepset.keras")

if MLFLOW_OK:
    mlflow.end_run()
    _run = mlflow.start_run(run_name="temporal_deepset_production")
    mlflow.log_params(HP)
    mlflow.log_params({
        "focal_fn"   : FOCAL_FN.__name__,
        "huber_delta": HP["HUBER_DELTA"],
    })
    _rid = _run.info.run_id
    log.info("MLflow run: %s", _rid)
else:
    _rid = None


class MLflowEpochCB(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if MLFLOW_OK and logs:
            for k, v in logs.items():
                mlflow.log_metric(k, float(v), step=epoch)


callbacks = [
    keras.callbacks.EarlyStopping(
        monitor=MONITOR, patience=12, min_delta=1e-4,
        restore_best_weights=True, mode="max", verbose=1),
    keras.callbacks.ModelCheckpoint(
        filepath=CKPT_PATH, monitor=MONITOR,
        save_best_only=True, mode="max", verbose=1),
    # ReduceLROnPlateau removed — incompatible with LearningRateSchedule;
    # it overwrites the schedule object with a scalar, freezing LR at 0.
    # Decay is already handled by WarmUpCosineDecay.
    keras.callbacks.TensorBoard(
        log_dir=str(PATHS["tb_logs"]),
        histogram_freq=0, write_graph=True, update_freq="epoch"),
    keras.callbacks.CSVLogger(
        str(PATHS["metrics"] / "training_log.csv"), append=False),
    MLflowEpochCB(),
]

log.info("Monitor: %s | ckpt: %s", MONITOR, CKPT_PATH)
log.info("Train: %d  Val: %d  Batch: %d  MaxEpoch: %d",
         len(y_tr), len(y_va), HP["BATCH_SIZE"], HP["EPOCHS"])

history = model.fit(
    ds_tr,
    validation_data = ds_va,
    epochs          = HP["EPOCHS"],
    callbacks       = callbacks,
    verbose         = 1,
)

best_ep    = int(np.argmax(history.history[MONITOR])) + 1
best_score = float(max(history.history[MONITOR]))
log.info("Done — best %s=%.4f @ epoch %d", MONITOR, best_score, best_ep)

if MLFLOW_OK:
    mlflow.log_metrics({"best_val_cell_future_fut_top1": best_score, "best_epoch": float(best_ep)})

09:20:31 │ INFO     │ Monitor: val_cell_future_fut_top1 | ckpt: /home/wassimmchichi/Downloads/Handover_projects/models/best_st_deepset.keras
09:20:31 │ INFO     │ Train: 20160  Val: 4320  Batch: 64  MaxEpoch: 60


Epoch 1/60
  6/315 [..............................] - ETA: 20s - loss: 0.9861 - cell_future_loss: 0.3085 - cell_now_loss: 0.3071 - sig_future_loss: 0.4135 - cell_future_fut_top1: 0.1969 - cell_now_top1: 0.2214 - cell_now_top3: 0.5260 - sig_future_sig_mae: 0.8001WARNING:tensorflow:Callback method `on_train_batch_end` is slow compared to the batch time (batch time: 0.0413s vs `on_train_batch_end` time: 0.0429s). Check your callbacks.
09:20:41 │ WARNING  │ Callback method `on_train_batch_end` is slow compared to the batch time (batch time: 0.0413s vs `on_train_batch_end` time: 0.0429s). Check your callbacks.
315/315 [==============================] - ETA: 0s - loss: 0.6633 - cell_future_loss: 0.2587 - cell_now_loss: 0.1051 - sig_future_loss: 0.2723 - cell_future_fut_top1: 0.5044 - cell_now_top1: 0.8146 - cell_now_top3: 0.9345 - sig_future_sig_mae: 0.5955
Epoch 1: val_cell_future_fut_top1 improved from -inf to 0.52819, saving model to /home/wassimmchichi/Downloads/Handover_projects/models/

## Section 9 · Training Curves

In [40]:
# ─── Section 9 · Training Curves ─────────────────────────────────────────────

hist = history.history
ep   = range(1, len(hist["loss"]) + 1)

# Define which metric pairs to plot
plot_specs = [
    ("loss",               "val_loss",               "Total Loss",           False),
    ("cell_now_top1",      "val_cell_now_top1",       "Now Top-1 Acc",        True),
    ("cell_now_top3",      "val_cell_now_top3",       "Now Top-3 Acc",        True),
    ("cell_future_fut_top1","val_cell_future_fut_top1","Future Top-1 Acc",    True),
    ("sig_future_sig_mae", "val_sig_future_sig_mae",  "Signal MAE (norm)",    False),
]

n_plots = len(plot_specs)
fig, axes = plt.subplots(1, n_plots, figsize=(4.5 * n_plots, 4.5))

for ax, (tr_k, va_k, title, higher_better) in zip(axes, plot_specs):
    if tr_k not in hist:
        ax.text(0.5, 0.5, f"{tr_k}\nnot found", ha="center", va="center")
        continue
    ax.plot(ep, hist[tr_k], lw=2, label="train")
    ax.plot(ep, hist[va_k], lw=2, ls="--", label="val")
    fn     = np.argmax if higher_better else np.argmin
    be, bv = fn(hist[va_k]) + 1, (max if higher_better else min)(hist[va_k])
    ax.axvline(be, color="red", ls=":", lw=1.2, alpha=0.8)
    ax.scatter([be], [bv], color="red", zorder=5, s=70,
               label=f"best @ ep {be} ({bv:.4f})")
    ax.set(title=title, xlabel="Epoch")
    ax.legend(fontsize=7); ax.grid(alpha=0.4)

fig.suptitle("Spatio-Temporal DeepSet — Training History", fontsize=13, fontweight="bold")
plt.tight_layout()
_out = PATHS["metrics"] / "training_curves.png"
plt.savefig(str(_out), dpi=150, bbox_inches="tight"); plt.close()
log.info("Saved: %s", _out)
if MLFLOW_OK:
    mlflow.log_artifact(str(_out))

09:25:43 │ INFO     │ Saved: /home/wassimmchichi/Downloads/Handover_projects/metrics/Temporal_deepset_production/training_curves.png


## Section 10 · Evaluation

### Autoregressive multi-step evaluation procedure
The model is evaluated **without teacher forcing** (production mode):
- `cell_now` → same as v1: top-1 / top-3 / top-5 accuracy
- `cell_future` → top-1 accuracy per step (s=1…5) and average  
- `sig_future` → MAE per feature (rsrp, sinr, load) in **denormalised** units

For evaluation we pass **zeros** as `teacher_signals` — the decoder's teacher-forcing branch is only active when `training=True` (inside `model.fit`). At `model.predict` time the layer uses the autoregressive path.

In [41]:
# ─── Section 10 · Evaluation ──────────────────────────────────────────────────

# ── 10a. Load best checkpoint ─────────────────────────────────────────────────
_BEST = str(PATHS["models"] / "best_st_deepset.keras")
log.info("Loading best checkpoint: %s", _BEST)

model = keras.models.load_model(
    _BEST,
    safe_mode=False,
    custom_objects={
        "MaskedGlobalAveragePooling": MaskedGlobalAveragePooling,
        "AutoregressiveDecoder"     : AutoregressiveDecoder,
        "WarmUpCosineDecay"         : WarmUpCosineDecay,
        FOCAL_FN.__name__           : FOCAL_FN,
        TEMP_FOCAL_FN.__name__      : TEMP_FOCAL_FN,
        HUBER_FN.__name__           : HUBER_FN,
    },
)

# ── 10b. Predict on test set ──────────────────────────────────────────────────
# teacher_signals in ds_te are the ground truth but the model IGNORES them
# at inference (training=False → autoregressive path in AutoregressiveDecoder).
preds_te = model.predict(ds_te, verbose=1)
# preds_te is a dict: {"cell_now": (N,K), "cell_future": (N,S,K), "sig_future": (N,S,3)}

probs_now  = preds_te["cell_now"]     # (N, K)
probs_fut  = preds_te["cell_future"]  # (N, S, K)
sig_pred   = preds_te["sig_future"]   # (N, S, 3) normalised

# ── 10c. Current-step metrics ─────────────────────────────────────────────────
y_pred_now = probs_now.argmax(axis=1)
top1_now   = float((y_pred_now == y_te).mean())
top3_now   = float(top_k_accuracy_score(y_te, probs_now, k=3, labels=ALL_LABELS))
top5_now   = float(top_k_accuracy_score(y_te, probs_now, k=5, labels=ALL_LABELS))

# ── 10d. Future-step cell accuracy (per step and averaged) ────────────────────
S = HP["PRED_STEPS"]
step_top1 = []
for s in range(S):
    y_pred_s = probs_fut[:, s, :].argmax(axis=1)        # (N,)
    acc_s    = float((y_pred_s == yfc_te[:, s]).mean())
    step_top1.append(acc_s)
avg_fut_top1 = float(np.mean(step_top1))

# ── 10e. Signal MAE in denormalised units ─────────────────────────────────────
# Inverse-transform predictions and ground truth back to original scale.
N     = len(y_te)
sig_pred_denorm = sig_scaler.inverse_transform(
    sig_pred.reshape(-1, 3)).reshape(N, S, 3)
sig_true_denorm = sig_scaler.inverse_transform(
    ys_te.reshape(-1, 3)).reshape(N, S, 3)
mae_per_feat    = np.abs(sig_pred_denorm - sig_true_denorm).mean(axis=(0, 1))  # (3,)
# mae_per_feat[0]=rsrp_mae, [1]=sinr_mae, [2]=load_mae

# ── Print results ─────────────────────────────────────────────────────────────
print("=" * 70)
print("  TEST RESULTS — Spatio-Temporal DeepSet (held-out UEs, best ckpt)")
print("=" * 70)
print(f"  Current-step cell selection:")
print(f"    Top-1 : {top1_now:.4f}  ({top1_now*100:.2f}%)")
print(f"    Top-3 : {top3_now:.4f}  ({top3_now*100:.2f}%)")
print(f"    Top-5 : {top5_now:.4f}  ({top5_now*100:.2f}%)")
print()
print(f"  Future cell selection (autoregressive, S={S} steps):")
for s, acc in enumerate(step_top1, 1):
    print(f"    Step t+{s} Top-1 : {acc:.4f}  ({acc*100:.2f}%)")
print(f"    Mean Top-1    : {avg_fut_top1:.4f}  ({avg_fut_top1*100:.2f}%)")
print()
print(f"  Signal regression MAE (denormalised):")
print(f"    RSRP  : {mae_per_feat[0]:.3f} dBm")
print(f"    SINR  : {mae_per_feat[1]:.3f} dB")
print(f"    Load  : {mae_per_feat[2]:.4f} (fraction)")
print()
print(classification_report(
    y_te, y_pred_now,
    labels=ALL_LABELS,
    target_names=[f"Cell {i}" for i in ALL_LABELS],
    digits=4, zero_division=0,
))

if MLFLOW_OK:
    mlflow.log_metrics({
        "test_now_top1"  : top1_now,
        "test_now_top3"  : top3_now,
        "test_now_top5"  : top5_now,
        "test_fut_avg_top1" : avg_fut_top1,
        "test_rsrp_mae"  : float(mae_per_feat[0]),
        "test_sinr_mae"  : float(mae_per_feat[1]),
        "test_load_mae"  : float(mae_per_feat[2]),
        **{f"test_fut_top1_s{s+1}": float(v) for s, v in enumerate(step_top1)},
    })

09:25:43 │ INFO     │ Loading best checkpoint: /home/wassimmchichi/Downloads/Handover_projects/models/best_st_deepset.keras
68/68 [==============================] - 3s 22ms/step
  TEST RESULTS — Spatio-Temporal DeepSet (held-out UEs, best ckpt)
  Current-step cell selection:
    Top-1 : 1.0000  (100.00%)
    Top-3 : 1.0000  (100.00%)
    Top-5 : 1.0000  (100.00%)

  Future cell selection (autoregressive, S=5 steps):
    Step t+1 Top-1 : 0.6498  (64.98%)
    Step t+2 Top-1 : 0.5794  (57.94%)
    Step t+3 Top-1 : 0.5426  (54.26%)
    Step t+4 Top-1 : 0.5229  (52.29%)
    Step t+5 Top-1 : 0.5120  (51.20%)
    Mean Top-1    : 0.5613  (56.13%)

  Signal regression MAE (denormalised):
    RSRP  : 5.556 dBm
    SINR  : 5.145 dB
    Load  : 0.0716 (fraction)

              precision    recall  f1-score   support

      Cell 0     1.0000    1.0000    1.0000       473
      Cell 1     1.0000    1.0000    1.0000       412
      Cell 2     1.0000    1.0000    1.0000       452
      Cell 3     1.00

## Section 11 · Confusion Matrix & Future-Step Accuracy

In [42]:
# ─── Section 11 · Confusion Matrix & Future-Step Accuracy Plots ───────────────

# ── 11a. Confusion matrix for current-step predictions ────────────────────────
C  = HP["MAX_CELLS"]
cl = [f"C{i}" for i in range(C)]
cm      = confusion_matrix(y_te, y_pred_now, labels=list(range(C)))
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=[f"P-{l}" for l in cl],
            yticklabels=[f"T-{l}" for l in cl],
            linewidths=0.5, ax=axes[0], annot_kws={"size": 9})
axes[0].set(title="Confusion Matrix (current step)", ylabel="Actual", xlabel="Predicted")

sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="YlGn",
            xticklabels=[f"P-{l}" for l in cl],
            yticklabels=[f"T-{l}" for l in cl],
            linewidths=0.5, ax=axes[1], vmin=0, vmax=1,
            annot_kws={"size": 9})
axes[1].set(title="Normalised Recall per Row", ylabel="Actual", xlabel="Predicted")
plt.tight_layout()
_out = PATHS["metrics"] / "confusion_matrix.png"
plt.savefig(str(_out), dpi=150, bbox_inches="tight"); plt.close()
log.info("Saved: %s", _out)

# ── 11b. Per-cell recall bar ───────────────────────────────────────────────────
per_recall = cm_norm.diagonal()
fig, ax = plt.subplots(figsize=(9, 3.5))
bars = ax.bar(range(C), per_recall,
              color=["#2196F3" if v >= 0.5 else "#F44336" for v in per_recall],
              edgecolor="white", linewidth=1)
ax.axhline(top1_now, color="black", ls="--", lw=1.2,
           label=f"Overall Top-1 ({top1_now:.3f})")
for b, v in zip(bars, per_recall):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.015,
            f"{v:.2f}", ha="center", va="bottom", fontsize=8)
ax.set(xticks=range(C), xticklabels=[f"Cell {i}" for i in range(C)],
       ylabel="Recall", ylim=(0, 1.18),
       title="Per-Cell Recall — Spatio-Temporal DeepSet")
ax.legend(); ax.grid(axis="y", alpha=0.4)
plt.xticks(rotation=30, ha="right"); plt.tight_layout()
_out = PATHS["metrics"] / "per_cell_recall.png"
plt.savefig(str(_out), dpi=150, bbox_inches="tight"); plt.close()
log.info("Saved: %s", _out)

# ── 11c. Future-step accuracy decay plot ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
steps_x = list(range(1, S + 1))
ax.plot(steps_x, [top1_now] + step_top1[:-1], "o--", color="#9E9E9E",
        label="Top-1 (incl. current)", lw=1.5)
ax.bar(steps_x, step_top1, alpha=0.7, color="#2196F3", edgecolor="white")
for x, v in zip(steps_x, step_top1):
    ax.text(x, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)
ax.axhline(avg_fut_top1, color="red", ls="--", lw=1.5,
           label=f"Mean fut Top-1 ({avg_fut_top1:.3f})")
ax.set(xlabel="Future Step s", ylabel="Top-1 Accuracy",
       title="Autoregressive Forecast Accuracy per Step",
       xticks=steps_x, xticklabels=[f"t+{s}" for s in steps_x],
       ylim=(0, 1.1))
ax.legend(); ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
_out = PATHS["metrics"] / "future_step_accuracy.png"
plt.savefig(str(_out), dpi=150, bbox_inches="tight"); plt.close()
log.info("Saved: %s", _out)

# ── 11d. Signal MAE per step ──────────────────────────────────────────────────
feat_names = ["RSRP (dBm)", "SINR (dB)", "Load"]
step_mae   = np.abs(sig_pred_denorm - sig_true_denorm).mean(axis=0)  # (S, 3)
fig, axes  = plt.subplots(1, 3, figsize=(14, 4))
for feat_i, (ax, fname) in enumerate(zip(axes, feat_names)):
    ax.bar(steps_x, step_mae[:, feat_i], color="#FF5722", alpha=0.8)
    ax.set(title=f"{fname} MAE per step", xlabel="Step", ylabel="MAE")
    ax.set_xticks(steps_x); ax.set_xticklabels([f"t+{s}" for s in steps_x])
    ax.grid(axis="y", alpha=0.4)
plt.suptitle("Signal Regression MAE — Autoregressive Forecasts",
             fontsize=12, fontweight="bold")
plt.tight_layout()
_out = PATHS["metrics"] / "signal_mae_per_step.png"
plt.savefig(str(_out), dpi=150, bbox_inches="tight"); plt.close()
log.info("Saved: %s", _out)

if MLFLOW_OK:
    for p in PATHS["metrics"].glob("*.png"):
        mlflow.log_artifact(str(p))

09:25:49 │ INFO     │ Saved: /home/wassimmchichi/Downloads/Handover_projects/metrics/Temporal_deepset_production/confusion_matrix.png
09:25:49 │ INFO     │ Saved: /home/wassimmchichi/Downloads/Handover_projects/metrics/Temporal_deepset_production/per_cell_recall.png
09:25:50 │ INFO     │ Saved: /home/wassimmchichi/Downloads/Handover_projects/metrics/Temporal_deepset_production/future_step_accuracy.png
09:25:50 │ INFO     │ Saved: /home/wassimmchichi/Downloads/Handover_projects/metrics/Temporal_deepset_production/signal_mae_per_step.png


## Section 12 · Save Metadata & Final Model

In [43]:
# ─── Section 12 · Save Artifacts ──────────────────────────────────────────────

FINAL_PATH = str(PATHS["models"] / "st_deepset_final.keras")
model.save(FINAL_PATH)
log.info("Final model: %s", FINAL_PATH)

# Save scalers for inference pipeline
with open(str(PATHS["models"] / "cell_scaler.pkl"), "wb") as f:
    pickle.dump(cell_scaler, f)
with open(str(PATHS["models"] / "sig_scaler.pkl"), "wb") as f:
    pickle.dump(sig_scaler, f)
log.info("Scalers saved.")

meta = {
    "created"           : datetime.datetime.now().isoformat(),
    "notebook"          : "notebooks/modeling/01_temporal_deepset_production.ipynb",
    "architecture"      : "SpatioTemporalDeepSet",
    "best_checkpoint"   : _BEST,
    "final_model"       : FINAL_PATH,
    "test_now_top1"     : round(top1_now, 4),
    "test_now_top3"     : round(top3_now, 4),
    "test_now_top5"     : round(top5_now, 4),
    "test_fut_avg_top1" : round(avg_fut_top1, 4),
    "test_rsrp_mae"     : round(float(mae_per_feat[0]), 4),
    "test_sinr_mae"     : round(float(mae_per_feat[1]), 4),
    "test_load_mae"     : round(float(mae_per_feat[2]), 4),
    "step_fut_top1"     : {f"t+{s+1}": round(v, 4) for s, v in enumerate(step_top1)},
    "best_val_score"    : round(best_score, 4),
    "best_epoch"        : best_ep,
    "loss_functions"    : {
        "cell_now"    : FOCAL_FN.__name__,
        "cell_future" : TEMP_FOCAL_FN.__name__,
        "sig_future"  : HUBER_FN.__name__,
    },
    "hyperparams"       : HP,
    "paths"             : {k: str(v) for k, v in PATHS.items()},
    "timing"            : {
        "sample_rate_s" : 0.2,
        "obs_s"         : HP["OBS_STEPS"] * 0.2,
        "forecast_s"    : HP["PRED_STEPS"] * 0.2,
    },
}
_meta_out = PATHS["metrics"] / "st_deepset_metadata.json"
json.dump(meta, open(str(_meta_out), "w"), indent=2)
log.info("Metadata: %s", _meta_out)

if MLFLOW_OK:
    mlflow.log_artifact(str(_meta_out))
    mlflow.tensorflow.log_model(
        model,
        artifact_path="st_deepset_keras",
        registered_model_name="spatiotemporal_deepset",
    )
    mlflow.end_run()
    log.info("MLflow run closed.")

# Artifact inventory
print()
print("=" * 70)
print("  ARTIFACT INVENTORY")
print("=" * 70)
for lbl, d in [("models/", PATHS["models"]), ("metrics/", PATHS["metrics"])]:
    print(f"\n  {lbl}")
    for p in sorted(Path(d).iterdir()):
        if p.is_file():
            print(f"    {p.name:<46s} {p.stat().st_size/1024:>7.1f} KB")
print()
print(f"  Current-step: Top-1 {top1_now:.4f} | Top-3 {top3_now:.4f} | Top-5 {top5_now:.4f}")
print(f"  Future mean Top-1 (t+1…t+5): {avg_fut_top1:.4f}")
print()
print("  TensorBoard:")
print(f"    tensorboard --logdir ../../tb_logs/")
print("  MLflow UI:")
print(f"    mlflow ui --backend-store-uri file://$(pwd)/../../mlflow/mlruns")

09:25:50 │ INFO     │ Final model: /home/wassimmchichi/Downloads/Handover_projects/models/st_deepset_final.keras
09:25:50 │ INFO     │ Scalers saved.
09:25:50 │ INFO     │ Metadata: /home/wassimmchichi/Downloads/Handover_projects/metrics/Temporal_deepset_production/st_deepset_metadata.json

  ARTIFACT INVENTORY

  models/
    6g_predictive_final.keras                        729.9 KB
    best_6g_predictive.keras                        9115.8 KB
    best_honet_final.keras                          3281.3 KB
    best_honet_p1.keras                             1545.1 KB
    best_honet_p2.keras                             2516.3 KB
    best_mh_transformer.keras                       3407.2 KB
    best_mtl_transformer.keras                      2096.9 KB
    best_set_transformer.keras                      2650.9 KB
    best_st_deepset.keras                           9277.2 KB
    best_strategic_deepset.keras                    2404.2 KB
    best_temporal_deepset.keras                      764

## Section 13 · Architecture Notes & Design Decisions

### Why this design fixes the flat-head failure

The previous v1 model predicted only the **current** optimal cell. A naïve extension
(`Dense(5 × MAX_CELLS)`) produces 5 independent softmaxes with no temporal coupling.
That fails because:

1. The cell selected at `t+1` constrains which cells are reachable at `t+2` (handover
   delay ≈ 50–200 ms, hysteresis prevents immediate switch-back).
2. The RSRP/SINR of the serving cell at `t+1` influences the triggering condition at
   `t+2` — a flat head cannot propagate this.

The **autoregressive decoder** (LSTMCell unrolled S=5 times with shared weights)
propagates `h_s` forward at each step, encoding the accumulated forecast trajectory.
At step s the decoder knows what cell was (softly) selected and what signals were
predicted in all prior steps.

### Teacher forcing during training
Without teacher forcing, early decoder states are random and gradients vanish before
reaching the encoder. With teacher forcing, the decoder is given correct signals
at each step during training, making the LSTM learn in a much more stable regime.
At inference, the autoregressive path is used.

### Tensor shapes — quick reference
```
Stage                 Tensor           Shape
────────────────────  ───────────────  ─────────────────────────
Input cells           inp_cells        (B, K=10, T=25, F=3)
Input mask            inp_mask         (B, K=10)
Input teacher         inp_teacher      (B, S=5,  3)
After TD-LSTM         trend            (B, K, 64)
After Phi MLP         phi              (B, K, 64)
Masked avg pool       z                (B, 64)
Max pool              max_phi          (B, 64)
Encoder context       ctx              (B, 128)
After rho MLP         rho              (B, K, 64)
Current logits        logits           (B, K)
Current probs         cell_now         (B, K)    → Output 1
Decoder h,c seeds     h_0,c_0          (B, 128)
Per-step input        dec_in           (B, K+3+128)
Per-step projected    dec_in_proj      (B, 128)
Per-step hidden       h_s              (B, 128)
Attend concat         attend_in        (B, K, 64+128)
Cell scores           scores           (B, K)
Cell probs (step s)   cell_probs_s     (B, K)
Weighted phi          selected_phi     (B, 64)
Signal pred (step s)  signals_s        (B, 3)
Stacked future cells  cell_future      (B, S=5, K) → Output 2
Stacked future sigs   sig_future       (B, S=5, 3) → Output 3
```

### Preprocessing summary
| Column group | Type | Parsing | Normalisation |
|---|---|---|---|
| `nb_rsrps`, `nb_sinrs`, `nb_loads` | `str` (list) | `parse_nb_array` → `(K,)` float | `StandardScaler` (cell_scaler) per `(K, T, F)` |
| `nb_cell_ids` | `str` (list) | `parse_nb_ids` → `(K,)` int | identity (used as lookup only) |
| `optimal_cell_rsrp/sinr/load` | scalar float | `.fillna(0)` | `StandardScaler` (sig_scaler) per `(S, 3)` |
| `nb_cell_types`, `nb_net_types` | `str` (list) | Not used in v2 model (no heterogeneous cell type embedding yet) | — |

### Recommended next experiments
1. **Scheduled sampling**: anneal from pure teacher forcing → pure autoregressive over epochs.
2. **Cell-type embedding**: add a learned embedding for `nb_cell_types` (macro/micro/pico) to `cell_feat`.
3. **Positional features**: add `nb_dists_m` and `nb_path_losses_db` as features (F=5).
4. **Decoder attention over history**: instead of just encoder summary, let the decoder attend over all T encoder hidden states (full seq2seq with attention).
5. **Scheduled sampling** ratio sweep via a custom callback.